In [ ]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt

%matplotlib inline
%matplotlib notebook

## 1.Load file.

### 1.1 Load csv file using 'pandas.read_csv'

In [ ]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [ ]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

### 1.2 Load csv file using ''csv'.

In [ ]:
import csv
with open('input_csv.csv') as f:
    M = list(csv.reader(f, delimiter=','))
    #reader = csv.reader(f, delimiter=',')
    #for row in reader:
        #print(row)
print(M[20])

## 2. Paved roof ###

### 2.1 Try ''while' iterator (with arrays)

In [ ]:
iters = np.shape(P_atm)[0] # total timestep.

In [ ]:
PR_measure = 0 # we do not consider 'measure' for the time being.

If using np.savetxt() to save csv file, and save all results after simulation.

In [ ]:
# This one can write result vertically in csv file 
IntStor_PR = np.zeros((iters,1))
E_atm_PR = np.zeros((iters,1))
Int_PR = np.zeros((iters,1))
R_swds_PR = np.zeros((iters,1))
R_mss_PR = np.zeros((iters,1))
R_measure_PR = np.zeros((iters,1)) # Not consider for the time being, all zeros.

IntStorCap_PR = 1.6  # InstorCap is defined as 1.6mm.  ----- parameter
Disc_partial_PR = 0  # Disconnected part is defined as 0%. ---- parameter
Storm_partial_PR = 1.0 # Stormwater part is defined as 100% ---- parameter
Mix_partial_PR = 1 - Storm_partial_PR

t = 1
while t <= iters-1:  
    # for PR I,E, Istor component only.
    Int_PR[t] = np.minimum(IntStorCap_PR, np.maximum(0, P_atm[t] + IntStor_PR[t-1]))
    E_atm_PR[t] = np.minimum(E_pot_OW[t], Int_PR[t])
    IntStor_PR[t] = Int_PR[t] - E_atm_PR[t] # This part is the same as the excel, different from the description file.
    R_swds_PR[t] = Storm_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    R_mss_PR[t] = Mix_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    t += 1
#print(E_atm_PR)
#print(R_swds_PR)
np.savetxt('sol/IntStor_sol.csv', (IntStor_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/E_atm_PR_sol.csv', (E_atm_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/Int_PR_sol.csv', (Int_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/R_swds_PR.csv', (R_swds_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/R_mss_PR.csv', (R_swds_PR), fmt='%.18e', delimiter=',')

In [ ]:
# This one can write three results horizontally in csv file.
IntStor_PR = np.zeros(iters)
E_atm_PR = np.zeros(iters)
Int_PR = np.zeros(iters)
R_swds_PR = np.zeros(iters)
R_mss_PR = np.zeros(iters)
R_measure_PR = np.zeros(iters) # Not consider for the time being, all zeros.

IntStorCap_PR = 1.6  # InstorCap is defined as 1.6mm.  ----- parameter
Disc_partial_PR = 0  # Disconnected part is defined as 0%. ---- parameter
Storm_partial_PR = 1.0 # Stormwater part is defined as 100% ---- parameter
Mix_partial_PR = 1 - Storm_partial_PR

t = 1
while t <= iters-1:  
    Int_PR[t] = np.minimum(IntStorCap_PR, np.maximum(0, P_atm[t] + IntStor_PR[t-1]))
    E_atm_PR[t] = np.minimum(E_pot_OW[t], Int_PR[t])
    IntStor_PR[t] = Int_PR[t] - E_atm_PR[t] # This part is the same as the excel, different from the description file.
    R_swds_PR[t] = Storm_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    R_mss_PR[t] = Mix_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    t += 1
#print(np.reshape(E_atm_PR,(iters,1)))
np.savetxt('sol/results.csv', (Int_PR, E_atm_PR, IntStor_PR, R_swds_PR, R_mss_PR), fmt='%.18e', delimiter=',')

If using q = [], and append().

If using csv_writerow to save csv file. Can we write down results after each time step? 

### 2.2 Try objective oriented programming (oop) ---Class

Class Function below solves for the solutions at time level t, based on the input(P and E) at time level t and output at time level t -1.

### Original code.

In [ ]:
class PavedRoof:
    def __init__(self, t, P_atm , E_pot_OW, Init_InStor, IntStorCap_PavedRoof = 1.6, StormFrac_PavedRoof = 1.0, DiscFrac_PavedRoof = 0.0):
        self.P_atm = P_atm
        self.init_InStor = Init_InStor 
        self.E_pot_OW = E_pot_OW
        self.IntStorCap = IntStorCap_PavedRoof
        self.StormFrac = StormFrac_PavedRoof
        self.MxdFrac = 1 - StormFrac_PavedRoof
        self.DiscFrac = DiscFrac_PavedRoof

    def __repr__(self):
        return 'Current P is ' + str(self.P_atm) + 'Current E is ' + str(self.E_pot_OW) + '.These are input information.'
        
    def sol(self):
        Int = np.minimum(self.IntStorCap, np.maximum(0, self.P_atm + self.init_InStor))
        E_atm = np.minimum( self.E_pot_OW, Int)
        IntStor = Int - E_atm
        R_swds = self.StormFrac * (1 - self.DiscFrac) * np.maximum(0, self.P_atm - E_atm -( IntStor - self.init_InStor) )
        R_mss = self.MxdFrac * (1 - self.DiscFrac) * np.maximum(0, self.P_atm - E_atm - ( IntStor - self.init_InStor))
        R_up = self.DiscFrac * np.maximum(0, self.P_atm - E_atm - (IntStor - self.init_InStor))
        return Int, E_atm, IntStor, R_swds, R_mss, R_up

In [ ]:
# Taking time step t = 8 as an example.
t1 = PavedRoof(8, 0.508, 0.04987013, 1.596675325)
print(t1.sol())
# The results correspondes with the excel solution.

Next, use 'while' to loop throughout the whole time steps, and try 2 different sets of coefficients to validate the class protocol.

Coefficient set 1: Default settings: IntStorCap_PavedRoof = 1.6, StormFrac_PavedRoof = 1.0, DiscFrac_PavedRoof = 0.0

Coefficient set 2: IntStorCap_PavedRoof = 1.3, StormFrac_PavedRoof = 0.5, DiscFrac_PavedRoof = 0.5

In [ ]:
t = 1
E_atm = [0]
Int = [0]
IntStor = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

while t <= iters-1:
    if t == 1:
        Init_InStor = 0
    m = PavedRoof(t, P_atm[t], E_pot_OW[t], Init_InStor, IntStorCap_PavedRoof = 1.3, StormFrac_PavedRoof = 0.5, DiscFrac_PavedRoof = 0.5)
    sol = m.sol()
    Int.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_swds.append(sol[3])
    R_mss.append(sol[4])
    R_up.append(sol[5])
    Init_InStor = sol[2]
    # print('time step', t)
    t += 1
filename = 'Class_results_Ori.csv'
np.savetxt('sol/' + filename, np.c_[Int, E_atm, IntStor, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure
# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
#date_column = pd.DataFrame({'Date': date})
#df = df.merge(date_column, left_index = True, right_index = True)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)
print('The results have been validated.')

### Modified code.
Original code is functioning and results have been validated with excel's. However there are several defficiencies of original code:

1. __np.minimum() to min()__. np.minimum() and min() behave differently when dealing with the arrays. However, they are equivalent when comparing two numbers. np.minimum() needs to call numpy package while min() is python internal function. So in the case of comparing numbers, min() could be more efficient for testing large datasets. 
2. __lowercase for variables and CamelCase for class name.__ It is a neat terminology in the professional programming.
3. __Differentiate the characteristics and input.__ \__init__(self, x, ...) is used in class to define the characteristics of objects, so x should be a attribute of the object. In our class case, the P_atm and E_pot_OW is the input which is independent of the PavedRoof class object, so they should be moved to sol fuunction. 
4. __Differentiate the state and parameters in \__init\__(self).__ It is easy to recognize lines of code if it is well-described.
5. __define a new function in class to calculate "mxd_frac".__ In original case, it should not a problem if we call the class for every "while" loop, becaues every time we call class, a new object is created and mxd_frac is calculated. However, it could become problematic if we only update not create class at each iteration. So it is good to take it into serious consideration.

In [ ]:
# np.minimum() to min()
# lowercase for variables and CamelCase for class name

class PavedRoof:
    def __init__(self, t, init_instor, intstorcap_pavedroof = 1.6, stormfrac_pavedroof = 1.0, discfrac_pavedroof = 0.0):
        
        # state
        self.init_instor = init_instor
        
        # parameters
        self.intstorcap = intstorcap_pavedroof
        self.stormfrac = stormfrac_pavedroof
        # self.MxdFrac = 1 - StormFrac_PavedRoof
        self.discfrac = discfrac_pavedroof
        
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are input information.'
        
    def mxd_frac(self):
        return 1 - self.stormfrac
        
    def sol(self, p_atm , e_pot_ow):
        intcp = min(self.intstorcap, max(0, p_atm + self.init_instor))
        e_atm = min(e_pot_ow, intcp)
        intstor = intcp - e_atm
        r_swds = self.stormfrac * (1 - self.discfrac) * max(0, p_atm - e_atm - (intstor - self.init_instor))
        r_mss = self.mxd_frac() * (1 - self.discfrac) * max(0, p_atm - e_atm - (intstor - self.init_instor))
        r_up = self.discfrac * max(0, p_atm - e_atm - (intstor - self.init_instor))
        
        # update state
        self.init_instor = intstor
        
        return intcp, e_atm, intstor, r_swds, r_mss, r_up

In [ ]:
t = 1
E_atm = [0]
Intcp = [0]
IntStor = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

# Give initial interception storage.
init_instor_t0 = 0 

m = PavedRoof(t, init_instor_t0, intstorcap_pavedroof = 1.3, stormfrac_pavedroof = 0.5, discfrac_pavedroof = 0.5)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t])
    
    Intcp.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_swds.append(sol[3])
    R_mss.append(sol[4])
    R_up.append(sol[5])
    
    # print('time step', t)
    t += 1
    
filename = 'Class_results_modified.csv'
np.savetxt('sol/' + filename, np.c_[Intcp, E_atm, IntStor, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure
# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
#date_column = pd.DataFrame({'Date': date})
#df = df.merge(date_column, left_index = True, right_index = True)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)
print('The results have been validated.')

### Conclusion.
The modified code looks much more neat than the original. 

Besides, modified structure is more efficient than the original code's structure since we do not create an object for each iteration, instead, we create in the beginning one object in which the state is updated by looping the internal function, and we iterate only sol() functions. 

This structure should work as a protocol for the other elements' buildup.